# Historical Image Generator

An AI image-generation pipeline combining language-model prompt enrichment with diffusion-based image synthesis.

The project uses Mistral-7B-Instruct to expand short prompts with historical and geographical context before generating images with DreamShaper-8.


In [ ]:
# HOW TO RUN:
# 1. Run installs/import, restart the runtime then run cells until Gradio
# 2. Run evaluation cells

In [ ]:
# ===== 1. Setup =====
# install + imports

In [ ]:
import sys

!{sys.executable} -m pip uninstall -y transformers diffusers accelerate bitsandbytes

!{sys.executable} -m pip install -U \
  transformers \
  diffusers \
  accelerate \
  bitsandbytes \
  gradio \
  typing_extensions

In [ ]:
import bitsandbytes, transformers, diffusers, accelerate

print("bitsandbytes:", bitsandbytes.__version__)
print("transformers:", transformers.__version__)
print("diffusers:", diffusers.__version__)
print("accelerate:", accelerate.__version__)

In [ ]:
# ===== 2. Load Models =====

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

llm_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(llm_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

text_model = AutoModelForCausalLM.from_pretrained(
    llm_name,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16
)

image_pipe = AutoPipelineForText2Image.from_pretrained(
    "Lykon/dreamshaper-8",
    dtype=torch.float16
).to("cuda")

image_pipe.enable_attention_slicing()
image_pipe.vae.enable_slicing()

print("text model device:", next(text_model.parameters()).device)
print("image pipe device:", image_pipe.device)

In [ ]:
# Patch DreamShaper dtype issue
image_pipe = image_pipe.to("cuda")

image_pipe.unet.to(dtype=torch.float32)
image_pipe.vae.to(dtype=torch.float32)

if hasattr(image_pipe, "text_encoder") and image_pipe.text_encoder is not None:
    image_pipe.text_encoder.to(dtype=torch.float32)

print("patched image pipe device:", image_pipe.device)

In [ ]:
# ===== 3. Prompt Functions =====

In [ ]:
import re
import torch

def baseline_prompt(subject, era, location, extra=""):
    subject = subject.strip()
    era = era.strip()
    location = location.strip()
    extra = extra.strip()

    if not subject:
        raise ValueError("Subject cannot be empty.")

    parts = [subject]
    if era:
        parts.append(era)
    if location:
        parts.append(location)
    if extra:
        parts.append(extra)

    return ", ".join(parts)


def expand_prompt(subject, era, location, extra=""):
    subject = subject.strip()
    era = era.strip()
    location = location.strip()
    extra = extra.strip()

    if not subject:
        raise ValueError("Subject cannot be empty.")

    subject = re.sub(r"^\s*portrait of an?\s+", "", subject, flags=re.IGNORECASE)
    subject = re.sub(r"^\s*portrait of\s+", "", subject, flags=re.IGNORECASE)
    subject = re.sub(r"^\s*portrait\s+", "", subject, flags=re.IGNORECASE)

    extra_text = extra if extra else "none"

    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict prompt compiler. "
                "You transform structured fields into one short image prompt. "
                "You do not invent objects, scenes, actions, or props."
            )
        },
        {
            "role": "user",
            "content": f"""
TASK:
Generate exactly one short visual sentence for an image model.
STRICT RULES:
- Output exactly one sentence
- Maximum 22 words
- Subject must stay the same subject
- Do not reinterpret the subject into a painting, wall object, statue, room scene, or different role
- Include subject, era, location, and extra if provided
- Use only visible physical details
- Use exactly one main verb
- Use only these connectors: with, in, on, under

NO INVENTION:
- Do not add any new object not directly supported by subject, era, location, or extra
- Do not add candles, lanterns, walls, screens, books, scrolls, pedestals, forests, laboratories, birds, weapons, or rooms unless explicitly required
- Do not add extra actions like painting, hanging, holding, looking, reflecting

FIELD RULES:
- Subject: keep it literal
- Era: convert into visible clothing, materials, or architecture details
- Location: convert into 1-2 visible environment details
- Extra: make it visibly present
- If extra is abstract, translate it only into visible weather or lighting, not objects

GOOD EXAMPLE:
Input:
Subject: warrior
Era: Mongol Empire
Location: Japan
Extra: snowy night with lanterns

Output:
A Mongol warrior in fur armor stands in Japan under snow with glowing lanterns, close-up, cinematic lighting

GOOD EXAMPLE:
Input:
Subject: scientist
Era: Renaissance
Location: Antarctica
Extra: scary

Output:
A scientist in long Renaissance robes stands in Antarctic snow under dark shadows, full body, realistic photo

BAD EXAMPLE:
A warrior portrait hangs on a wall in a Japanese room with lanterns

Why bad:
- changes subject into wall art
- invents room and wall

BAD EXAMPLE:
A scientist paints in an Antarctic laboratory with candles

Why bad:
- invents laboratory and candles
- invents action

OUTPUT FORMAT:
[subject + era detail + verb + location + extra + composition + style]

INPUT:
Subject: {subject}
Era: {era}
Location: {location}
Extra: {extra_text}

Output:
"""

        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(text_model.device)

    outputs = text_model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    expanded = decoded[len(prompt):].strip() if decoded.startswith(prompt) else decoded.strip()

    for marker in ["[/INST]", "[INST]", "<s>", "</s>"]:
        expanded = expanded.replace(marker, "")

    if "Output:" in expanded:
        expanded = expanded.split("Output:")[-1].strip()

    expanded = expanded.split("\n")[0].strip().strip(",")

    return expanded


In [ ]:
# ===== 4. Image Generation =====

In [ ]:
def generate_image(subject, era, location, extra="", expand=True):
    if expand:
        final_prompt = expand_prompt(subject, era, location, extra)
    else:
        final_prompt = baseline_prompt(subject, era, location, extra)

    negative_prompt = (
        "blurry, low quality, text, watermark, cropped, duplicate elements, "
        "distorted composition, modern objects"
    )

    with torch.no_grad():
        image = image_pipe(
            final_prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=25,
            guidance_scale=7.5,
            num_images_per_prompt=1
        ).images[0]

    return image, final_prompt

In [ ]:
# ===== 5. Gradio UI =====

In [ ]:
# NOTE:
# To use the UI, run all cells and then run the final Gradio cell.
# A temporary public link will be generated automatically.

import gradio as gr

def app_generate(subject, era, location, extra):
    image, expanded_prompt = generate_image(
        subject=subject,
        era=era,
        location=location,
        extra=extra,
        expand=True
    )
    return image, expanded_prompt

demo = gr.Interface(
    fn=app_generate,
    inputs=[
        gr.Textbox(label="Subject", placeholder="e.g. warrior standing"),
        gr.Textbox(label="Historical Era", placeholder="e.g. Ancient Egypt"),
        gr.Textbox(label="Location", placeholder="e.g. New York city street"),
        gr.Textbox(label="Extra", placeholder="e.g. flying sand, rain, lanterns"),
    ],
    outputs=[
        gr.Image(label="Generated Image"),
        gr.Textbox(label="Expanded Prompt"),
    ],
    title="Historical-Geographical Image Generator",
    description=(
        "Enter a subject, historical era, location, and extra visual condition. "
        "The system expands the prompt with Mistral, then generates an image using DreamShaper."
    ),
    examples=[
        ["warrior standing", "Aztec Empire", "Venice canal", "smoke rising"],
        ["scholar writing", "Islamic Golden Age", "Tokyo subway station", "crowded rush hour"],
        ["spaceship landing", "Ancient Rome", "Antarctica", "snow falling"],
        ["digital computer glowing", "Digital Age", "Ancient Greek temple", "blue light"],
    ]
)

demo.launch(share=True)

In [ ]:
# ===== 6. Evaluation Tests =====

In [ ]:
eval_tests = [
    # People
    ("portrait of a dancer", "Ancient Egypt", "New York city street", "flying sand"),
    ("scholar writing", "Islamic Golden Age", "Tokyo subway station", "crowded rush hour"),
    ("leader sitting", "Indus Valley Civilization", "Paris street", "rain falling"),
    ("factory worker standing", "Industrial Revolution", "Amazon rainforest", "heavy rain"),
    ("astronaut walking", "Cold War", "Sahara desert", "dust storm"),
    ("musician performing", "Baroque era", "Seoul concert hall", "bright stage lights"),
    ("doctor standing", "Victorian era", "Dubai hospital lobby", "blue light"),
    ("warrior standing", "Aztec Empire", "Venice canal", "smoke rising"),
    ("merchant walking", "Mughal Empire", "Iceland coastline", "strong wind"),
    ("teacher standing", "Ancient China", "London skyline", "thick fog"),
    ("sailor rowing", "Viking Age", "Pacific Ocean coast", "storm waves"),

    # Objects
    ("spaceship landing", "Ancient Rome", "Antarctica", "snow falling"),
    ("printing press displayed", "Renaissance", "Himalayas", "heavy snowfall"),
    ("digital computer glowing", "Digital Age", "Ancient Greek temple", "blue light"),
    ("boat sailing", "Polynesian seafaring period", "Amazon river", "thick fog"),
]

In [ ]:
# Run generation + save outputs - safer version
import os
import csv
import gc
import torch
from IPython.display import display

output_root = "actuall_eval_outputs"
os.makedirs(output_root, exist_ok=True)

csv_path = os.path.join(output_root, "evaluation_metadata.csv")

with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)

    writer.writerow([
        "test_id",
        "subject",
        "era",
        "location",
        "extra",
        "baseline_prompt",
        "expanded_prompt",
        "baseline_image_path",
        "expanded_image_path",
        "status"
    ])

    for i, (subject, era, location, extra) in enumerate(eval_tests, 1):
        test_folder = os.path.join(output_root, f"test_{i:02d}")
        os.makedirs(test_folder, exist_ok=True)

        print("=" * 80)
        print(f"TEST {i}")
        print(subject, era, location, extra)

        try:
            torch.cuda.empty_cache()
            gc.collect()

            baseline_img, baseline_prompt_text = generate_image(
                subject, era, location, extra, expand=False
            )

            torch.cuda.empty_cache()
            gc.collect()

            expanded_img, expanded_prompt_text = generate_image(
                subject, era, location, extra, expand=True
            )

            baseline_path = os.path.join(test_folder, "baseline.png")
            expanded_path = os.path.join(test_folder, "expanded.png")
            prompts_path = os.path.join(test_folder, "prompts.txt")

            baseline_img.save(baseline_path)
            expanded_img.save(expanded_path)

            with open(prompts_path, "w", encoding="utf-8") as f:
                f.write(f"Subject: {subject}\n")
                f.write(f"Era: {era}\n")
                f.write(f"Location: {location}\n")
                f.write(f"Extra: {extra}\n\n")
                f.write("Baseline prompt:\n")
                f.write(baseline_prompt_text + "\n\n")
                f.write("Expanded prompt:\n")
                f.write(expanded_prompt_text + "\n")

            writer.writerow([
                i,
                subject,
                era,
                location,
                extra,
                baseline_prompt_text,
                expanded_prompt_text,
                baseline_path,
                expanded_path,
                "success"
            ])

            print("Saved to:", test_folder)
            display(baseline_img)
            display(expanded_img)

        except Exception as e:
            print("FAILED:", e)

            writer.writerow([
                i,
                subject,
                era,
                location,
                extra,
                "",
                "",
                "",
                "",
                f"failed: {e}"
            ])

            torch.cuda.empty_cache()
            gc.collect()
            continue

print("Done.")
print("Saved metadata to:", csv_path)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

steps = [
    "User Input\n(subject, era, location, extra)",
    "Text Expansion\n(Mistral 7B Instruct)",
    "Expanded Prompt",
    "Image Generation\n(DreamShaper-8)",
    "Final Output\n(generated image)"
]

fig, ax = plt.subplots(figsize=(12, 3))
ax.set_xlim(0, 12)
ax.set_ylim(0, 3)
ax.axis("off")

x_positions = [0.5, 3.0, 5.4, 7.6, 10.0]
box_width = 1.8
box_height = 0.9
y = 1.1

for i, (x, text) in enumerate(zip(x_positions, steps)):
    box = FancyBboxPatch(
        (x, y),
        box_width,
        box_height,
        boxstyle="round,pad=0.08",
        linewidth=1.5,
        facecolor="white",
        edgecolor="black"
    )
    ax.add_patch(box)
    ax.text(
        x + box_width / 2,
        y + box_height / 2,
        text,
        ha="center",
        va="center",
        fontsize=9
    )

    if i < len(steps) - 1:
        arrow = FancyArrowPatch(
            (x + box_width, y + box_height / 2),
            (x_positions[i + 1], y + box_height / 2),
            arrowstyle="->",
            mutation_scale=15,
            linewidth=1.5,
            color="black"
        )
        ax.add_patch(arrow)

plt.title("System Pipeline", fontsize=14, weight="bold")
plt.tight_layout()

plt.savefig("system_pipeline_flowchart.png", dpi=300, bbox_inches="tight")
plt.show()